# 3. Distributions et espaces de Sobolev

**Objectif** : comprendre pourquoi les fonctions ordinaires ne suffisent pas pour les EDP,
introduire les dérivées au sens des distributions et construire les espaces H¹ et H² numériquement.

## Plan
1. Dérivée au sens des distributions : la fonction valeur absolue
2. La masse de Dirac comme distribution non-régulière
3. Espace de Sobolev H¹(0,1) : norme et vérification numérique
4. Traces : restriction au bord pour une fonction H¹
5. H¹₀(0,1) : espace avec conditions aux limites de Dirichlet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── 1. Dérivée au sens des distributions ─────────────────────────────────────
# f(x) = |x| sur [-1, 1] : non différentiable en 0 au sens classique
# Mais f'(x) = sign(x) au sens des distributions (= dérivée faible)
#
# Vérification numérique :
#   Pour tout phi in C_c^inf([-1,1]), <f', phi> = -<f, phi'> = -int f*phi' dx
#   D'autre part int sign(x)*phi(x) dx doit donner la même valeur.

x = np.linspace(-1, 1, 4000)
dx = x[1] - x[0]

# Fonction test phi : bump function C_c^inf à support dans (-0.8, 0.8)
def bump(x, a=0.8):
    out = np.zeros_like(x)
    mask = np.abs(x) < a
    t = x[mask] / a
    out[mask] = np.exp(-1.0 / (1.0 - t**2))
    # Normalisation
    return out / (out.max() + 1e-300)

phi  = bump(x)
dphi = np.gradient(phi, dx)

f       = np.abs(x)        # f(x) = |x|
f_prime = np.sign(x)       # dérivée faible attendue

# <f', phi>_distrib = -int f * phi' dx
lhs = -np.trapezoid(f * dphi, x)
# int sign(x) * phi dx
rhs = np.trapezoid(f_prime * phi, x)

print('Dérivée au sens des distributions de |x| :')
print(f'  -int |x| phi\'(x) dx = {lhs:.8f}')
print(f'   int sign(x) phi(x) dx = {rhs:.8f}')
print(f'  Erreur = {abs(lhs-rhs):.2e}  => sign(x) est bien la dérivée faible de |x|')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x, f, label='f(x) = |x|')
axes[0].plot(x, f_prime, '--', label="f'(x) = sign(x)  (sens distrib.)")
axes[0].set_title('Dérivée faible de |x|')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(x, phi, label='φ (fonction test)')
axes[1].plot(x, dphi, '--', label="φ'")
axes[1].set_title('Fonction test φ ∈ C_c^∞(−0.8, 0.8)')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2. Espace de Sobolev H¹(0,1) ──────────────────────────────────────────────
#
# H¹(0,1) = { u ∈ L²(0,1) : u' ∈ L²(0,1) au sens des distributions }
# Norme : ||u||²_{H¹} = ||u||²_{L²} + ||u'||²_{L²}
#
# On teste trois fonctions :
#   (a) u(x) = sin(pi*x)              -> u ∈ H¹  (lisse)
#   (b) u(x) = sqrt(x)                -> u ∈ H¹  (u'= 1/(2sqrt(x)) ∈ L²)
#   (c) u(x) = x^(-0.4)               -> u ∉ H¹  (u'= -0.4 x^{-1.4} ∉ L²)

x = np.linspace(1e-6, 1, 10000)  # éviter x=0
dx_val = x[1] - x[0]

functions = [
    ('sin(πx)',   np.sin(np.pi * x),  np.pi * np.cos(np.pi * x),  True),
    ('√x',        np.sqrt(x),          0.5 / np.sqrt(x),            True),
    ('x^{-0.4}', x**(-0.4),           -0.4 * x**(-1.4),            False),
]

print(f'{"Fonction":<15} {"||u||_L2":>12} {"||u\' ||_L2":>12} {"||u||_H1":>12} {"∈ H¹?"}' )
print('-' * 60)
for name, u, du, in_H1 in functions:
    norm_L2_u  = np.sqrt(np.trapezoid(u**2, x))
    norm_L2_du = np.sqrt(np.trapezoid(np.minimum(du**2, 1e8), x))  # clip pour éviter inf
    norm_H1    = np.sqrt(norm_L2_u**2 + np.trapezoid(du**2, x) if in_H1
                         else float('inf'))
    in_str = '✓' if in_H1 else '✗  (||u\'||_L2 = +∞)'
    print(f'{name:<15} {norm_L2_u:>12.4f} {norm_L2_du:>12.4f} {"∞" if not in_H1 else f"{np.sqrt(norm_L2_u**2+norm_L2_du**2):.4f}":>12} {in_str}')

In [ ]:
# ── 3. H¹₀(0,1) : fonctions H¹ à trace nulle au bord ─────────────────────────
#
# H¹₀(0,1) = { u ∈ H¹(0,1) : u(0) = u(1) = 0 }
#
# Inégalité de Poincaré : il existe C > 0 tel que ||u||_{L²} ≤ C ||u'||_{L²}
# pour tout u ∈ H¹₀(0,1). Sur [0,1] la constante optimale est C = 1/π.
#
# On vérifie numériquement sur une famille de fonctions tests.

x  = np.linspace(0, 1, 5000)
dx = x[1] - x[0]

# Fonctions dans H¹₀ : u_k(x) = sin(k*pi*x), k ∈ {1,...,10}
Poincare_ratios = []
for k in range(1, 11):
    u  = np.sin(k * np.pi * x)
    du = k * np.pi * np.cos(k * np.pi * x)
    norm_u   = np.sqrt(np.trapezoid(u**2,  x))
    norm_du  = np.sqrt(np.trapezoid(du**2, x))
    ratio = norm_u / norm_du
    Poincare_ratios.append((k, ratio))

ks, ratios = zip(*Poincare_ratios)
plt.figure(figsize=(7, 4))
plt.plot(ks, ratios, 'o-', color='steelblue', label=r'$\|u_k\|_{L^2} / \|u_k\|_{H^1_0}$')
plt.axhline(1/np.pi, color='tomato', ls='--', label=r'Constante de Poincaré $C = 1/\pi$')
plt.xlabel('k')
plt.title("Inégalité de Poincaré sur $H^1_0(0,1)$")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Constante 1/pi = {1/np.pi:.6f}')
print('Le ratio ||u_k||_L2 / ||u_k\'||_L2 est maximisé pour k=1 et vaut 1/pi.')

## Récapitulatif

| Espace | Définition | Norme | Utilité pour les EDP |
|--------|-----------|-------|----------------------|
| $L^2(\Omega)$ | Fonctions de carré intégrable | $\|u\|_{L^2}$ | Second membre |
| $H^1(\Omega)$ | $u \in L^2$, $\nabla u \in L^2$ | $\|u\|_{L^2} + \|\nabla u\|_{L^2}$ | Solution faible de Poisson |
| $H^1_0(\Omega)$ | $H^1$ avec trace nulle au bord | idem | Conditions de Dirichlet |
| $H^2(\Omega)$ | $u \in H^1$, $D^2 u \in L^2$ | $\|u\|_{H^1} + \|D^2 u\|_{L^2}$ | Régularité elliptique |

**Inégalité de Poincaré** : sur $H^1_0(\Omega)$, la semi-norme $\|\nabla u\|_{L^2}$ est une norme équivalente,
ce qui est crucial pour démontrer la coercivité dans Lax-Milgram (notebook 4).